# Diagnóstico de calidad de datos – Top 50 Spotify Colombia 2024

| | |
| --- | --- |
| **Evidencia** | GA1-220501094-01-AA1-EV06 – Diagnosticar la calidad de datos: preparación, exploración y documentación |
| **Programa** | Tecnólogo en Análisis y Desarrollo de Software (228118) – SENA |
| **Aprendiz** | [Tu nombre] |
| **Instructor** | Carlos Enrique Navia Torres |

**Pregunta orientadora:** ¿Qué condiciones de calidad presentan los conjuntos de datos seleccionados para mi proyecto y qué acciones de preparación o limpieza deberían considerarse antes de iniciar su análisis?

## Momento 1. Reconocimiento del proyecto y de los datasets

**Nombre del proyecto:** Géneros musicales en el Top 50 de Spotify Colombia 2024.

**Propósito:** identificar qué estilos musicales dominan lo que más se escucha en Colombia, para que artistas, emisoras y organizadores de eventos tomen decisiones basadas en datos.

**Pregunta analítica (EV02):** ¿Qué géneros musicales aparecen con más frecuencia en el Top 50 diario de Spotify Colombia durante 2024?

**Datasets seleccionados (EV03):**

| Dataset | Archivo | Fuente | Qué contiene | Relación con el problema |
| --- | --- | --- | --- | --- |
| Dataset 1 | `universal_top_spotify_songs.csv` | Kaggle – Top Spotify Songs in 73 Countries | Top 50 diario de más de 70 países, con posición, artistas, fecha y popularidad | Fuente principal: de aquí salen las canciones de Colombia en 2024 |
| Dataset 2 | `regional-co-weekly-2024-01-11.csv` | Spotify Charts (fuente oficial) | Top 200 semanal de Colombia, semana del 11 de enero de 2024 | Validación: comprobar que el Dataset 1 coincide con la lista oficial |

**Aspectos de calidad que debo revisar antes de usar los datos:**
1. **Valores faltantes:** columnas con datos vacíos o valores que representan "sin dato".
2. **Tipos de datos:** que las fechas sean fechas y los números sean números.
3. **Duplicados:** filas o canciones repetidas.
4. **Rangos:** posiciones fuera de 1–50 (o 1–200) y popularidad fuera de 0–100.
5. **Cobertura:** que estén los 366 días de 2024 para Colombia.
6. **Disponibilidad del dato clave:** si existe una columna de género.

## Momento 2. Preparación del entorno, repositorio y carga de datos

**Entorno:** Python 3.12 en un entorno virtual (`venv`), creado y activado desde Git Bash en VS Code.

```bash
python -m venv venv
source venv/Scripts/activate
pip install -r requirements.txt
```

**Dependencias (`requirements.txt`):** `pandas`, `notebook`, `ipykernel`.

**Estructura del repositorio:**

```
proyecto-spotify-colombia/
├── README.md
├── .gitignore
├── requirements.txt
├── data/raw/            # datos originales sin modificar
├── notebooks/
│   └── diagnostico_calidad.ipynb
└── reports/
    └── informe_diagnostico.md
```

El archivo de Kaggle no se sube a GitHub por su tamaño (está en `.gitignore`). En el `README.md` se explica cómo descargarlo.

In [2]:
import pandas as pd

pd.set_option("display.max_columns", None)   # muestra todas las columnas
print("Versión de pandas:", pd.__version__)

Versión de pandas: 3.1.0


In [3]:
# Rutas: el notebook está en notebooks/, por eso se sube un nivel con ../
RUTA_KAGGLE = "../data/raw/universal_top_spotify_songs.csv"
RUTA_CHARTS = "../data/raw/regional-co-weekly-2024-01-11.csv"

# Cada dataset en un DataFrame independiente con un nombre que lo identifica
kaggle = pd.read_csv(RUTA_KAGGLE, low_memory=False)   # Dataset 1
charts = pd.read_csv(RUTA_CHARTS)                     # Dataset 2

In [4]:
# Verificación de la carga
print("Dataset 1 – Kaggle cargado:", kaggle.shape[0], "filas y", kaggle.shape[1], "columnas")
print("Dataset 2 – Spotify Charts cargado:", charts.shape[0], "filas y", charts.shape[1], "columnas")

Dataset 1 – Kaggle cargado: 2110316 filas y 25 columnas
Dataset 2 – Spotify Charts cargado: 200 filas y 9 columnas


## Momento 3. Inspección de la estructura de los datasets

Para cada DataFrame se ejecutan los 8 comandos de la guía:

| Comando | Propósito |
| --- | --- |
| `head()` | Visualizar los primeros registros |
| `tail()` | Visualizar los últimos registros |
| `shape` | Identificar el número de filas y columnas |
| `columns` | Consultar los nombres de las variables |
| `dtypes` | Reconocer los tipos de datos |
| `info()` | Examinar la estructura y los valores no nulos |
| `describe()` | Estadísticas descriptivas de las variables numéricas |
| `describe(include='object')` | Describir las variables de tipo texto |

### 3.1 Dataset 1 – Kaggle (Top 50 diario por país)

In [5]:
# head(): primeros registros
kaggle.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,duration_ms,album_name,album_release_date,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,2RkZ5LkEzeHGRsmDqKwmaJ,Ordinary,Alex Warren,1,1,0,NaN,2025-06-11,95,False,186964,"You'll Be Alright, Kid (Chapter 1)",2024-09-26,0.368,0.694,2,-6.141,1,0.0600,0.704000,0.000007,0.0550,0.391,168.115,3
1,42UBPzRMh5yyz0EDPr6fr1,Manchild,Sabrina Carpenter,2,-1,48,NaN,2025-06-11,89,True,213645,Manchild,2025-06-05,0.731,0.685,7,-5.087,1,0.0572,0.122000,0.000000,0.3170,0.811,123.010,4
2,0FTmksd2dxiE5e3rWyJXs6,back to friends,sombr,3,0,1,NaN,2025-06-11,98,False,199032,back to friends,2024-12-27,0.436,0.723,1,-2.291,1,0.0301,0.000094,0.000088,0.0929,0.235,92.855,4
3,7so0lgd0zP2Sbgs2d7a1SZ,Die With A Smile,"Lady Gaga, Bruno Mars",4,0,-1,NaN,2025-06-11,91,False,251667,MAYHEM,2025-03-07,0.519,0.601,6,-7.727,0,0.0317,0.289000,0.000000,0.1260,0.498,157.964,3
4,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,5,1,0,NaN,2025-06-11,100,False,210373,HIT ME HARD AND SOFT,2024-05-17,0.747,0.507,2,-10.171,1,0.0358,0.200000,0.060800,0.1170,0.438,104.978,4


In [6]:
# tail(): últimos registros
kaggle.tail()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,duration_ms,album_name,album_release_date,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
2110311,0AYt6NMyyLd0rLuvr0UkMH,Slime You Out (feat. SZA),"Drake, SZA",46,4,0,AE,2023-10-18,84,True,310490,For All The Dogs,2023-10-06,0.483,0.408,5,-9.243,0,0.0502,0.5080,0.000000,0.259,0.105,88.880,3
2110312,2Gk6fi0dqt91NKvlzGsmm7,SAY MY GRACE (feat. Travis Scott),"Offset, Travis Scott",47,3,0,AE,2023-10-18,80,True,173253,SET IT OFF,2023-10-13,0.773,0.635,10,-5.060,1,0.0452,0.0585,0.000000,0.132,0.476,121.879,4
2110313,26b3oVLrRUaaybJulow9kz,People,Libianca,48,2,0,AE,2023-10-18,88,False,184791,People,2022-12-06,0.573,0.422,10,-7.621,0,0.0678,0.5510,0.000013,0.102,0.693,124.357,5
2110314,5ydjxBSUIDn26MFzU3asP4,Rainy Days,V,49,1,0,AE,2023-10-18,88,False,179560,Layover,2023-08-11,0.633,0.454,9,-8.016,0,0.0875,0.7390,0.000000,0.148,0.282,74.828,4
2110315,59NraMJsLaMCVtwXTSia8i,Prada,"cassö, RAYE, D-Block Europe",50,0,0,AE,2023-10-18,94,True,132359,Prada,2023-08-11,0.638,0.717,8,-5.804,1,0.0375,0.0010,0.000002,0.113,0.422,141.904,4


In [7]:
# shape: número de filas y columnas
kaggle.shape

(2110316, 25)

In [8]:
# columns: nombres de las variables
kaggle.columns

Index(['spotify_id', 'name', 'artists', 'daily_rank', 'daily_movement',
       'weekly_movement', 'country', 'snapshot_date', 'popularity',
       'is_explicit', 'duration_ms', 'album_name', 'album_release_date',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'time_signature'],
      dtype='str')

In [9]:
# dtypes: tipos de datos
kaggle.dtypes

spotify_id                str
name                      str
artists                   str
daily_rank              int64
daily_movement          int64
weekly_movement         int64
country                   str
snapshot_date             str
popularity              int64
is_explicit              bool
duration_ms             int64
album_name                str
album_release_date        str
danceability          float64
energy                float64
key                     int64
loudness              float64
mode                    int64
speechiness           float64
acousticness          float64
instrumentalness      float64
liveness              float64
valence               float64
tempo                 float64
time_signature          int64
dtype: object

In [37]:
# info(): estructura y valores no nulos

kaggle.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 2110316 entries, 0 to 2110315
Data columns (total 25 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   spotify_id          2110316 non-null  str    
 1   name                2110286 non-null  str    
 2   artists             2110287 non-null  str    
 3   daily_rank          2110316 non-null  int64  
 4   daily_movement      2110316 non-null  int64  
 5   weekly_movement     2110316 non-null  int64  
 6   country             2081408 non-null  str    
 7   snapshot_date       2110316 non-null  str    
 8   popularity          2110316 non-null  int64  
 9   is_explicit         2110316 non-null  bool   
 10  duration_ms         2110316 non-null  int64  
 11  album_name          2109494 non-null  str    
 12  album_release_date  2109657 non-null  str    
 13  danceability        2110316 non-null  float64
 14  energy              2110316 non-null  float64
 15  key                 211031

In [11]:
# describe(): estadísticas de las variables numéricas
kaggle.describe()

,daily_rank,daily_movement,weekly_movement,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
count,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06,2.110316e+06
mean,2.549302e+01,9.230660e-01,2.932522e+00,7.590713e+01,1.943095e+05,6.759191e-01,6.488031e-01,5.526496e+00,-6.772439e+00,5.364571e-01,9.550408e-02,2.748491e-01,2.316185e-02,1.705588e-01,5.462535e-01,1.221008e+02,3.900368e+00
std,1.442900e+01,7.015535e+00,1.233129e+01,1.576302e+01,4.986054e+04,1.440469e-01,1.689142e-01,3.594979e+00,3.874849e+00,4.986692e-01,9.138830e-02,2.509297e-01,1.137297e-01,1.247528e-01,2.311527e-01,2.781938e+01,4.044448e-01
min,1.000000e+00,-4.900000e+01,-4.900000e+01,0.000000e+00,0.000000e+00,0.000000e+00,2.010000e-05,0.000000e+00,-5.434100e+01,0.000000e+00,0.000000e+00,3.450000e-06,0.000000e+00,1.390000e-02,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.300000e+01,-1.000000e+00,-3.000000e+00,6.500000e+01,1.626370e+05,5.800000e-01,5.520000e-01,2.000000e+00,-7.830000e+00,0.000000e+00,3.840000e-02,6.670000e-02,0.000000e+00,9.610000e-02,3.700000e-01,1.000190e+02,4.000000e+00
50%,2.500000e+01,0.000000e+00,0.000000e+00,7.900000e+01,1.861910e+05,7.000000e-01,6.680000e-01,6.000000e+00,-6.064000e+00,1.000000e+00,5.810000e-02,1.910000e-01,1.340000e-06,1.220000e-01,5.480000e-01,1.199650e+02,4.000000e+00
75%,3.800000e+01,2.000000e+00,5.000000e+00,8.800000e+01,2.187010e+05,7.800000e-01,7.670000e-01,9.000000e+00,-4.723000e+00,1.000000e+00,1.120000e-01,4.370000e-01,1.010000e-04,2.040000e-01,7.330000e-01,1.400210e+02,4.000000e+00
max,5.000000e+01,4.900000e+01,4.900000e+01,1.000000e+02,1.296000e+06,9.880000e-01,9.980000e-01,1.100000e+01,3.233000e+00,1.000000e+00,9.570000e-01,9.960000e-01,9.950000e-01,9.830000e-01,9.920000e-01,2.360890e+02,5.000000e+00


In [38]:
# describe(include='object'): descripción de las variables de texto
kaggle.describe(include=["object", "str"])

,spotify_id,name,artists,country,snapshot_date,album_name,album_release_date
count,2110316,2110286,2110287,2081408,2110316,2109494,2109657
unique,24983,21809,13726,72,583,16440,2973
top,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,DO,2024-03-16,HIT ME HARD AND SOFT,2024-05-17
freq,16673,16673,32587,29176,3942,30813,34806


**Características principales del Dataset 1:**
- Tiene **2.110.316 filas** y **25 columnas**: es el Top 50 diario de 72 países más la lista Global.
- Tipos: 7 columnas de texto (`str`), 8 enteras (`int64`), 9 decimales (`float64`) y 1 booleana (`is_explicit`).
- `snapshot_date` y `album_release_date` están guardadas como **texto**, no como fecha.
- `info(show_counts=True)` muestra que `country` tiene **2.081.408 valores no nulos de 2.110.316**: le faltan 28.908, que corresponden a la lista Global.
- `head()` muestra la fecha **2025-06-11** y `tail()` la fecha **2023-10-18**: el archivo está ordenado de la fecha más reciente a la más antigua y cubre de octubre de 2023 a junio de 2025. Solo una parte corresponde a 2024.
- `describe()` muestra rangos válidos en `daily_rank` (1–50) y `popularity` (0–100), pero valores sospechosos: `duration_ms`, `tempo` y `time_signature` tienen **mínimo 0**, algo imposible en una canción real.
- `describe(include=...)` muestra **24.983 canciones únicas** (`spotify_id`) pero solo **21.809 nombres únicos**: hay canciones distintas con el mismo nombre (versiones, remixes), así que el nombre no sirve como identificador.
- **No existe una columna de género**, que es la variable central de la pregunta.

### 3.2 Dataset 2 – Spotify Charts (Top 200 semanal Colombia)

In [13]:
# head(): primeros registros
charts.head()

,rank,uri,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams
0,1,spotify:track:7bywjHOc0wSjGGbj04XbVi,"Feid, ATL Jacob",LUNA,UMLE - Latino,1,1,6,2498259
1,2,spotify:track:7iQXYTyuG13aoeHxGG28Nh,"Bad Bunny, Feid",PERRO NEGRO,Rimas Entertainment LLC.,1,2,13,1729841
2,3,spotify:track:05sO6FK8E9l2cfwnEUTjHq,"Feid, Pirlo",CUAL ES ESA,UMLE - Latino,2,3,6,1506198
3,4,spotify:track:0R6NfOiLzLj4O5VbYSJAjf,Xavi,La Diabla,Interscope Records,4,6,4,1490474
4,5,spotify:track:5RqSsdzTNPX1uzkmlHCFvK,"KAROL G, Peso Pluma",QLONA,Bichota/Interscope Records,1,4,22,1261274


In [14]:
# tail(): últimos registros
charts.tail()

,rank,uri,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams
195,196,spotify:track:1dw7qShk971xMD6r6mA4VN,"KAROL G, Quevedo",PERO TÚ,BICHOTA RECORDS / INTERSCOPE RECORDS / UNIVERS...,14,-1,43,203514
196,197,spotify:track:3eqCJfgJJs8iKx49KO12s3,Feid,REMIX EXCLUSIVO,UMLE - Latino,5,-1,40,202757
197,198,spotify:track:01ppKlDRCmPpusO3yNrSRY,"Bad Bunny, Luar La L",TELEFONO NUEVO,Rimas Entertainment LLC.,7,-1,9,202586
198,199,spotify:track:5HNjGi0qYXvZgksCYRVXAH,Alcolirykoz,Todo Lo Bueno Tarda,Alcolirykoz,108,-1,46,202504
199,200,spotify:track:0goXAEnjWhssHU0y9Zyqyf,Feid,LUCES DE TECNO,UMLE - Latino,7,-1,13,202193


In [15]:
# shape: número de filas y columnas
charts.shape

(200, 9)

In [16]:
# columns: nombres de las variables
charts.columns

Index(['rank', 'uri', 'artist_names', 'track_name', 'source', 'peak_rank',
       'previous_rank', 'weeks_on_chart', 'streams'],
      dtype='str')

In [17]:
# dtypes: tipos de datos
charts.dtypes

rank              int64
uri                 str
artist_names        str
track_name          str
source              str
peak_rank         int64
previous_rank     int64
weeks_on_chart    int64
streams           int64
dtype: object

In [18]:
# info(): estructura y valores no nulos
charts.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   rank            200 non-null    int64
 1   uri             200 non-null    str  
 2   artist_names    200 non-null    str  
 3   track_name      200 non-null    str  
 4   source          200 non-null    str  
 5   peak_rank       200 non-null    int64
 6   previous_rank   200 non-null    int64
 7   weeks_on_chart  200 non-null    int64
 8   streams         200 non-null    int64
dtypes: int64(5), str(4)
memory usage: 14.2 KB


In [19]:
# describe(): estadísticas de las variables numéricas
charts.describe()

,rank,peak_rank,previous_rank,weeks_on_chart,streams
count,200.000000,200.000000,200.000000,200.000000,2.000000e+02
mean,100.500000,38.695000,71.145000,65.840000,3.853639e+05
std,57.879185,42.580364,63.723287,70.701328,2.869800e+05
min,1.000000,1.000000,-1.000000,1.000000,2.021930e+05
25%,50.750000,5.000000,6.750000,15.750000,2.319448e+05
50%,100.500000,20.000000,61.500000,41.000000,2.735820e+05
75%,150.250000,67.250000,124.500000,90.500000,4.125085e+05
max,200.000000,165.000000,200.000000,368.000000,2.498259e+06


In [40]:
   # describe(include='object'): descripción de las variables de texto
   
charts.describe(include=["object", "str"])

,uri,artist_names,track_name,source
count,200,200,200,200
unique,200,155,200,86
top,spotify:track:7bywjHOc0wSjGGbj04XbVi,Feid,LUNA,UMLE - Latino
freq,1,13,1,27


**Características principales del Dataset 2:**
- Tiene **200 filas** y **9 columnas**: es un Top 200 de una sola semana (11 de enero de 2024), no un Top 50 diario.
- 5 columnas son enteros (`rank`, `peak_rank`, `previous_rank`, `weeks_on_chart`, `streams`) y 4 son texto (`uri`, `artist_names`, `track_name`, `source`).
- `info()` muestra 200 valores no nulos en todas las columnas: **no hay nulos visibles**.
- `describe()` muestra que `previous_rank` tiene un **mínimo de -1**, un valor que no tiene sentido como posición.
- `streams` va de 202.193 a 2.498.259 reproducciones; la mediana es 273.582. La canción número 1 tiene unas 9 veces más reproducciones que la mediana.
- La disquera más frecuente (`top` de `source`) es "UMLE - Latino", con 27 canciones.
- Tampoco tiene columna de género.

### 3.3 Tabla de diagnóstico estructural

In [21]:
def resumen_tipos(df):
    conteo = df.dtypes.astype(str).value_counts()
    return ", ".join(f"{tipo}: {n}" for tipo, n in conteo.items())

tabla_estructural = pd.DataFrame({
    "Dataset": ["Dataset 1 – Kaggle (Top 50 diario)", "Dataset 2 – Spotify Charts (Top 200 semanal)"],
    "Filas": [kaggle.shape[0], charts.shape[0]],
    "Columnas": [kaggle.shape[1], charts.shape[1]],
    "Tipos de datos": [resumen_tipos(kaggle), resumen_tipos(charts)],
    "Celdas nulas": [int(kaggle.isna().sum().sum()), int(charts.isna().sum().sum())],
    "Observaciones": [
        "Fechas como texto; country con nulos (lista Global); varios artistas por celda; sin columna de género",
        "Sin nulos visibles; previous_rank usa -1; Top 200 de una sola semana; sin columna de género",
    ],
})
tabla_estructural

,Dataset,Filas,Columnas,Tipos de datos,Celdas nulas,Observaciones
0,Dataset 1 – Kaggle (Top 50 diario),2110316,25,"float64: 9, int64: 8, str: 7, bool: 1",30448,Fechas como texto; country con nulos (lista Gl...
1,Dataset 2 – Spotify Charts (Top 200 semanal),200,9,"int64: 5, str: 4",0,Sin nulos visibles; previous_rank usa -1; Top ...


## Momento 4. Diagnóstico inicial de calidad

Se revisan los cinco aspectos que pide la guía: valores faltantes, tipos de datos, posibles inconsistencias, limitaciones y acciones de limpieza.

### 4.1 Valores faltantes

In [22]:
def tabla_faltantes(df):
    t = pd.DataFrame({
        "nulos": df.isna().sum(),
        "porcentaje (%)": (df.isna().mean() * 100).round(2),
    })
    return t[t["nulos"] > 0].sort_values("nulos", ascending=False)

print("Dataset 1 – Kaggle:")
display(tabla_faltantes(kaggle))
print("Dataset 2 – Spotify Charts:")
display(tabla_faltantes(charts))

Dataset 1 – Kaggle:


,nulos,porcentaje (%)
country,28908,1.37
album_name,822,0.04
album_release_date,659,0.03
name,30,0.00
artists,29,0.00


Dataset 2 – Spotify Charts:


,nulos,porcentaje (%)


In [23]:
# Las filas sin país corresponden a la lista Global, no a un país
if "country" in kaggle.columns:
    sin_pais = kaggle[kaggle["country"].isna()]
    print("Filas sin país:", len(sin_pais))
    display(sin_pais.head(3))

Filas sin país: 28908


,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,duration_ms,album_name,album_release_date,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,2RkZ5LkEzeHGRsmDqKwmaJ,Ordinary,Alex Warren,1,1,0,NaN,2025-06-11,95,False,186964,"You'll Be Alright, Kid (Chapter 1)",2024-09-26,0.368,0.694,2,-6.141,1,0.0600,0.704000,0.000007,0.0550,0.391,168.115,3
1,42UBPzRMh5yyz0EDPr6fr1,Manchild,Sabrina Carpenter,2,-1,48,NaN,2025-06-11,89,True,213645,Manchild,2025-06-05,0.731,0.685,7,-5.087,1,0.0572,0.122000,0.000000,0.3170,0.811,123.010,4
2,0FTmksd2dxiE5e3rWyJXs6,back to friends,sombr,3,0,1,NaN,2025-06-11,98,False,199032,back to friends,2024-12-27,0.436,0.723,1,-2.291,1,0.0301,0.000094,0.000088,0.0929,0.235,92.855,4


**Faltantes disfrazados en el Dataset 2.** `previous_rank = -1` significa que la canción **no estaba en la lista la semana anterior**. `isna()` no lo detecta, pero es un dato ausente.

In [24]:
nuevas = (charts["previous_rank"] == -1).sum()
print("Canciones con previous_rank = -1:", nuevas)
print("Promedio de previous_rank con el -1:", round(charts["previous_rank"].mean(), 2))
print("Promedio de previous_rank sin el -1:", round(charts.loc[charts["previous_rank"] != -1, "previous_rank"].mean(), 2))

Canciones con previous_rank = -1: 44
Promedio de previous_rank con el -1: 71.14
Promedio de previous_rank sin el -1: 91.49




- **Dataset 1 (Kaggle):** las filas sin valor en `country` no son errores. Corresponden a la lista **Global**, que no pertenece a ningún país. Si no se filtran, se mezclarían con las canciones de Colombia y los conteos de géneros quedarían inflados con canciones que quizá no estuvieron en el Top 50 colombiano. Por eso, antes de analizar hay que quedarse solo con `country == "CO"`.

- **Dataset 2 (Spotify Charts):** no tiene nulos visibles, pero 44 canciones tienen `previous_rank = -1`. Ese `-1` no es una posición: significa que la canción no estaba en la lista la semana anterior, así que es un dato faltante disfrazado. Como pandas lo trata como un número, cambia los cálculos. El promedio de `previous_rank` da 71,14 con el `-1` incluido y 91,49 sin él. Si no se corrige, cualquier comparación entre semanas daría resultados equivocados. La solución es reemplazar el `-1` por `NaN`, para que pandas lo reconozca como valor ausente.

- **Conclusión:** revisar solo `isna()` no basta. También hay que buscar valores que representan "sin dato" aunque estén escritos como números.

### 4.2 Tipos de datos

In [25]:
# ¿Las fechas se pueden convertir? Se prueba sin modificar el DataFrame original
for col in ["snapshot_date", "album_release_date"]:
    if col in kaggle.columns:
        convertida = pd.to_datetime(kaggle[col], errors="coerce")
        invalidas = convertida.isna().sum() - kaggle[col].isna().sum()
        print(f"{col}: tipo actual = {kaggle[col].dtype} | valores que no se pueden convertir a fecha: {invalidas}")

snapshot_date: tipo actual = str | valores que no se pueden convertir a fecha: 0
album_release_date: tipo actual = str | valores que no se pueden convertir a fecha: 0


In [26]:
# ¿Los identificadores y textos tienen el tipo esperado?
print("Dataset 2 – tipos:")
print(charts.dtypes)

Dataset 2 – tipos:
rank              int64
uri                 str
artist_names        str
track_name          str
source              str
peak_rank         int64
previous_rank     int64
weeks_on_chart    int64
streams           int64
dtype: object


**Interpretación de los tipos de datos:**

*Dataset 1 – Kaggle*
- `snapshot_date` y `album_release_date` son de tipo `str`, pero representan **fechas**. Así no se puede filtrar por año ni agrupar por mes de forma confiable. La prueba con `pd.to_datetime(errors="coerce")` muestra **0 valores que no se pueden convertir**, así que la conversión es segura.
- `key`, `mode` y `time_signature` son `int64`, pero en realidad son **códigos de categoría** (por ejemplo, `mode` 1 = mayor y 0 = menor). El tipo sirve para guardarlos, pero no tiene sentido sacarles un promedio.
- `is_explicit` es `bool`: el tipo es correcto.
- `duration_ms` está en milisegundos. El tipo es correcto, pero conviene pasarlo a minutos para que se entienda mejor.

*Dataset 2 – Spotify Charts*
- `rank`, `peak_rank`, `weeks_on_chart` y `streams` son `int64`: correcto.
- `previous_rank` es `int64`, pero usa **-1 para indicar "sin dato"**. Debería ser un entero que acepte nulos (`Int64`) con `NaN` en lugar de -1.
- `uri` tiene el formato `spotify:track:ID`, mientras que en el Dataset 1 el identificador (`spotify_id`) es solo el `ID`. Para comparar los dos datasets hay que **extraer el ID** de `uri`.
- El Dataset 2 **no tiene columna de fecha**: la semana (2024-01-11) solo aparece en el nombre del archivo.

### 4.3 Posibles inconsistencias

**Duplicados**

In [27]:
print("Dataset 1 – filas completamente duplicadas:", kaggle.duplicated().sum())
if {"snapshot_date", "country", "daily_rank"}.issubset(kaggle.columns):
    print("Dataset 1 – misma fecha, país y posición repetidos:",
          kaggle.duplicated(subset=["snapshot_date", "country", "daily_rank"]).sum())

print("Dataset 2 – filas duplicadas:", charts.duplicated().sum())
print("Dataset 2 – canciones (uri) repetidas:", charts["uri"].duplicated().sum())

Dataset 1 – filas completamente duplicadas: 0
Dataset 1 – misma fecha, país y posición repetidos: 3686
Dataset 2 – filas duplicadas: 0
Dataset 2 – canciones (uri) repetidas: 0


In [34]:
# Origen de los duplicados: ¿misma canción repetida o canciones distintas en la misma posición?
CLAVE_LISTA = ["snapshot_date", "country", "daily_rank"]
filas_duplicadas = kaggle[kaggle.duplicated(subset=CLAVE_LISTA, keep=False)]

print("Filas involucradas en duplicados:", len(filas_duplicadas))
print("Duplicados con la misma canción:", kaggle.duplicated(subset=CLAVE_LISTA + ["spotify_id"]).sum())
print("Fechas afectadas:", sorted(filas_duplicadas["snapshot_date"].unique()))

filas_por_lista = kaggle.groupby(["snapshot_date", "country"], dropna=False).size()
print("Listas (fecha, país) con más de 50 canciones:", (filas_por_lista > 50).sum())

# Impacto en Colombia 2024
filas_co_2024 = kaggle[(kaggle["country"] == "CO") & kaggle["snapshot_date"].str.startswith("2024")]
filas_por_dia_co = filas_co_2024.groupby("snapshot_date").size()
print("\nColombia 2024 – duplicados:", filas_co_2024.duplicated(subset=CLAVE_LISTA).sum())
print("Colombia 2024 – días con más de 50 canciones:")
print(filas_por_dia_co[filas_por_dia_co > 50])

Filas involucradas en duplicados: 7372
Duplicados con la misma canción: 0
Fechas afectadas: ['2023-10-27', '2023-11-20', '2024-03-14', '2024-03-16', '2024-03-18', '2025-04-30']
Listas (fecha, país) con más de 50 canciones: 358

Colombia 2024 – duplicados: 33
Colombia 2024 – días con más de 50 canciones:
snapshot_date
2024-03-14    54
2024-03-16    55
2024-03-18    52
dtype: int64


**Interpretación de los duplicados del Dataset 1:**

- Ninguna fila está repetida completa, pero hay **3.686 casos** con la misma fecha, país y posición. Al incluir `spotify_id` en la comparación los duplicados bajan a **0**: no es la misma canción descargada dos veces, son **canciones distintas ocupando la misma posición**, algo imposible en un Top 50 real.
- El problema se concentra en **6 fechas** (2023-10-27, 2023-11-20, 2024-03-14, 2024-03-16, 2024-03-18 y 2025-04-30) y afecta a **358 listas** de distintos países, incluida la lista Global.
- **Impacto en Colombia 2024:** hay **33 posiciones repetidas** en tres días: 2024-03-14 (54 canciones), 2024-03-16 (55) y 2024-03-18 (52). Esos días solo tienen 11 canciones de más, pero 33 posiciones repetidas, lo que indica que también faltan posiciones: las listas están **mezcladas**.
- Estos días están pegados a dos días que faltan (2024-03-15 y 2024-03-17). Es probable que las listas de días cercanos se hayan guardado con la fecha equivocada.
- **Posible impacto:** si se cuentan los géneros con estos días, algunas canciones se cuentan de más y otras quedan fuera, así que la frecuencia de géneros se distorsiona.
- **Acción propuesta:** excluir del análisis los días 2024-03-14, 2024-03-16 y 2024-03-18 de Colombia y documentarlos como datos no confiables. Así quedarían **346 de 366 días** útiles.

**Valores fuera de rango**

In [28]:
if "daily_rank" in kaggle.columns:
    print("Dataset 1 – daily_rank fuera de 1–50:", (~kaggle["daily_rank"].between(1, 50)).sum())
if "popularity" in kaggle.columns:
    print("Dataset 1 – popularity fuera de 0–100:", (~kaggle["popularity"].between(0, 100)).sum())
if "duration_ms" in kaggle.columns:
    print("Dataset 1 – canciones con duración 0 o negativa:", (kaggle["duration_ms"] <= 0).sum())

print("Dataset 2 – rank fuera de 1–200:", (~charts["rank"].between(1, 200)).sum())
print("Dataset 2 – streams en 0 o negativos:", (charts["streams"] <= 0).sum())
print("Dataset 2 – peak_rank mayor que rank (imposible):", (charts["peak_rank"] > charts["rank"]).sum())

Dataset 1 – daily_rank fuera de 1–50: 0
Dataset 1 – popularity fuera de 0–100: 0
Dataset 1 – canciones con duración 0 o negativa: 29
Dataset 2 – rank fuera de 1–200: 0
Dataset 2 – streams en 0 o negativos: 0
Dataset 2 – peak_rank mayor que rank (imposible): 0


**Varios artistas en una misma celda y textos escritos de forma distinta**

In [41]:
if "artists" in kaggle.columns:
    print("Dataset 1 – filas con varios artistas:", kaggle["artists"].astype(str).str.contains(",").sum())
print("Dataset 2 – filas con varios artistas:", charts["artist_names"].str.contains(",").sum())

# La misma disquera escrita de formas distintas
print("\nFormas en que aparece la disquera Bichota en el Dataset 2:")
for valor in charts.loc[charts["source"].str.contains("bichota", case=False), "source"].unique():
    print("  -", valor)

Dataset 1 – filas con varios artistas: 858692
Dataset 2 – filas con varios artistas: 100

Formas en que aparece la disquera Bichota en el Dataset 2:
  - Bichota/Interscope Records
  - BICHOTA RECORDS / INTERSCOPE RECORDS / UNIVERSAL MUSIC LATINO


**Interpretación de las inconsistencias:**

- **Valores fuera de rango:** `daily_rank` (1–50), `popularity` (0–100), `rank` (1–200) y `streams` están dentro de rangos válidos. Sin embargo, en el Dataset 1 hay **29 canciones con `duration_ms` igual a 0**, lo cual es imposible. Coinciden con los mínimos en 0 de `tempo` y `time_signature` en `describe()`: probablemente son canciones a las que Spotify no les calculó las características de audio.
- **Varios artistas en una celda:** **858.692 de 2.110.316 filas** del Dataset 1 (40,7 %) y **100 de 200** del Dataset 2 (50 %) tienen varios artistas separados por coma. Para asignar un género por artista hay que separar esa columna. Si no se hace, `"Feid, ATL Jacob"` se trataría como un artista distinto de `"Feid"` y no tendría género asignado.
- **Riesgo al separar por coma:** algunos artistas llevan una coma en su propio nombre (por ejemplo, *Tyler, The Creator*). Separar sin revisar podría partir un artista en dos.
- **Nombres escritos de formas distintas:** la disquera de Karol G aparece como `Bichota/Interscope Records` y como `BICHOTA RECORDS / INTERSCOPE RECORDS / UNIVERSAL MUSIC LATINO`. Para pandas son **dos valores distintos**, así que cualquier conteo por disquera queda dividido. El mismo problema puede aparecer con nombres de artistas y géneros.
- **Conclusión:** ninguno de estos problemas aparece con `isna()`. Solo se detectan revisando rangos y el contenido del texto. Si no se corrigen, el conteo de géneros contaría artistas de más o los dejaría sin género.

### 4.4 Limitaciones de los datos

In [30]:
# Cobertura: la pregunta necesita los 366 días de 2024 para Colombia
if {"country", "snapshot_date"}.issubset(kaggle.columns):
    fechas = pd.to_datetime(kaggle["snapshot_date"], errors="coerce")
    print("Rango de fechas del Dataset 1:", fechas.min().date(), "a", fechas.max().date())

    co_2024 = kaggle[(kaggle["country"] == "CO") & (fechas.dt.year == 2024)]
    dias_presentes = pd.to_datetime(co_2024["snapshot_date"]).dt.normalize().unique()
    todos_los_dias = pd.date_range("2024-01-01", "2024-12-31", freq="D")
    dias_faltantes = todos_los_dias.difference(dias_presentes)

    print("Filas de Colombia en 2024:", len(co_2024), "(máximo esperado: 50 x 366 = 18.300)")
    print("Días con datos:", len(dias_presentes), "de 366")
    print("Días faltantes:", len(dias_faltantes))
    print(list(dias_faltantes.strftime("%Y-%m-%d"))[:30])

Rango de fechas del Dataset 1: 2023-10-18 a 2025-06-11
Filas de Colombia en 2024: 17461 (máximo esperado: 50 x 366 = 18.300)
Días con datos: 349 de 366
Días faltantes: 17
['2024-03-15', '2024-03-17', '2024-04-20', '2024-04-21', '2024-04-22', '2024-04-23', '2024-04-24', '2024-11-28', '2024-11-29', '2024-11-30', '2024-12-01', '2024-12-02', '2024-12-03', '2024-12-04', '2024-12-05', '2024-12-07', '2024-12-08']


In [31]:
# ¿Existe la columna de género, que es la variable central de la pregunta?
for nombre, df in [("Dataset 1 – Kaggle", kaggle), ("Dataset 2 – Spotify Charts", charts)]:
    tiene = any("genre" in c.lower() or "genero" in c.lower() for c in df.columns)
    print(f"{nombre}: ¿tiene columna de género? {'Sí' if tiene else 'No'}")

Dataset 1 – Kaggle: ¿tiene columna de género? No
Dataset 2 – Spotify Charts: ¿tiene columna de género? No


**Limitaciones identificadas:**
1. **Ningún dataset tiene la columna de género.** Sin ella no se puede responder la pregunta; hay que unir una tabla de géneros por artista (API de Spotify o Last.fm).
2. El Dataset 2 es **un Top 200 de una sola semana**, así que solo sirve para validar, no para el análisis anual.
3. Cobertura del Dataset 1 para Colombia en 2024: **[ ] de 366 días** _(completa con la salida de arriba)_.

### 4.5 Tabla de hallazgos y acciones de limpieza propuestas

In [32]:
def valor(condicion, texto_si_no):
    try:
        return condicion()
    except Exception:
        return texto_si_no

hallazgos = pd.DataFrame([
    ["Dataset 1 – Kaggle", "Valores faltantes en country",
     valor(lambda: f"{kaggle['country'].isna().sum()} filas sin país", "revisar"),
     "Si no se filtran, se mezclan con la lista Global",
     "Filtrar country == 'CO'"],
    ["Dataset 1 – Kaggle", "Fechas guardadas como texto",
     valor(lambda: f"snapshot_date es de tipo {kaggle['snapshot_date'].dtype}", "revisar"),
     "No se puede filtrar 2024 ni agrupar por mes",
     "Convertir con pd.to_datetime()"],
    ["Dataset 1 – Kaggle", "Varios artistas en una celda",
     valor(lambda: f"{kaggle['artists'].astype(str).str.contains(',').sum()} filas", "revisar"),
     "No se puede asignar un solo género por canción",
     "Separar con str.split(',') y tomar el artista principal"],
    ["Dataset 1 – Kaggle", "No hay columna de género",
     "Ninguna columna contiene 'genre'",
     "No se puede responder la pregunta solo con este archivo",
     "Unir con géneros de la API de Spotify o Last.fm (merge)"],
    ["Dataset 2 – Charts", "Faltante disfrazado en previous_rank",
     f"{(charts['previous_rank'] == -1).sum()} canciones con -1",
     "Distorsiona promedios y comparaciones",
     "Reemplazar -1 por NaN"],
    ["Dataset 2 – Charts", "Varios artistas en una celda",
     f"{charts['artist_names'].str.contains(',').sum()} de {len(charts)} canciones",
     "No se puede asignar un solo género",
     "Separar el artista principal"],
    ["Dataset 2 – Charts", "Disquera escrita de formas distintas",
     f"{charts.loc[charts['source'].str.contains('bichota', case=False), 'source'].nunique()} variantes de Bichota",
     "Los conteos por disquera quedan divididos",
     "Pasar a minúsculas y unificar nombres"],
    ["Dataset 2 – Charts", "Top 200 de una sola semana",
     f"{len(charts)} filas de una fecha",
     "No es comparable directamente con el Top 50 diario",
     "Filtrar rank <= 50 y usarlo solo para validar"],
], columns=["Dataset", "Problema identificado", "Evidencia encontrada", "Posible impacto", "Acción propuesta"])
hallazgos

,Dataset,Problema identificado,Evidencia encontrada,Posible impacto,Acción propuesta
0,Dataset 1 – Kaggle,Valores faltantes en country,28908 filas sin país,"Si no se filtran, se mezclan con la lista Global",Filtrar country == 'CO'
1,Dataset 1 – Kaggle,Fechas guardadas como texto,snapshot_date es de tipo str,No se puede filtrar 2024 ni agrupar por mes,Convertir con pd.to_datetime()
2,Dataset 1 – Kaggle,Varios artistas en una celda,858692 filas,No se puede asignar un solo género por canción,"Separar con str.split(',') y tomar el artista ..."
3,Dataset 1 – Kaggle,No hay columna de género,Ninguna columna contiene 'genre',No se puede responder la pregunta solo con est...,Unir con géneros de la API de Spotify o Last.f...
4,Dataset 2 – Charts,Faltante disfrazado en previous_rank,44 canciones con -1,Distorsiona promedios y comparaciones,Reemplazar -1 por NaN
5,Dataset 2 – Charts,Varios artistas en una celda,100 de 200 canciones,No se puede asignar un solo género,Separar el artista principal
6,Dataset 2 – Charts,Disquera escrita de formas distintas,2 variantes de Bichota,Los conteos por disquera quedan divididos,Pasar a minúsculas y unificar nombres
7,Dataset 2 – Charts,Top 200 de una sola semana,200 filas de una fecha,No es comparable directamente con el Top 50 di...,Filtrar rank <= 50 y usarlo solo para validar


In [33]:
# Guardar la tabla de hallazgos en reports/ para el informe técnico
hallazgos.to_csv("../reports/tabla_hallazgos.csv", index=False, encoding="utf-8-sig")
print("Tabla guardada en reports/tabla_hallazgos.csv")

Tabla guardada en reports/tabla_hallazgos.csv


## Momento 5. Conclusiones del diagnóstico

El informe técnico completo está en `reports/informe_diagnostico.md`. Estas son las conclusiones principales:

1. **Los datos se pueden usar, pero no tal como vienen.** La estructura es ordenada y la mayoría de los tipos son correctos.
2. **El problema más grave es que no hay columna de género.** Antes del análisis hay que construir y unir una tabla de géneros por artista.
3. **Hay problemas que no se ven a simple vista:** el `-1` en `previous_rank`, varios artistas en una celda y nombres escritos de formas distintas. Sin este diagnóstico habrían producido conteos equivocados.
4. **Siguiente paso:** filtrar Colombia 2024, convertir fechas, separar el artista principal y conseguir los géneros.

_[Agrega aquí tu conclusión sobre la cobertura de días de Colombia en 2024.]_

## Momento 6. Verificación, publicación y entrega

- [ ] El notebook se ejecuta completo con **Run All** sin errores.
- [ ] El diagnóstico incluye los dos datasets.
- [ ] El informe técnico (`reports/informe_diagnostico.md`) coincide con los resultados de este notebook.
- [ ] Las carpetas y archivos siguen la estructura del repositorio.
- [ ] `requirements.txt` y `README.md` explican cómo reproducir el trabajo.
- [ ] Los cambios están publicados en GitHub:

```bash
git add .
git commit -m "Diagnóstico de calidad de los datasets"
git push
```

- [ ] El repositorio es público o el instructor tiene acceso.
- [ ] La URL del repositorio está entregada en Google Classroom.